# Week 8: Case Study with Protein Language Models

Welcome to your Capstone Project! In all the modules from this course, you have learned the fundamentals of Python, built neural networks from scratch, and experimented with state-of-the-art Vision Transformers.

In this final module, we bring everything together to solve a fundamental problem in cell biology: **Subcellular Localization**.

## Learning objectives
- Understand transfer learning with embeddings
- Generate embeddings from pretrained models
- Implement multiclass classifier in `sklearn` and `PyTorch`
- Design and run comparative experiments on protein-related prediction tasks
- Analyze and interpret your results and draw conclusions based on a test set

### Library Imports

In [ ]:
# !pip install -q pandas numpy matplotlib seaborn scikit-learn torch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Part 1: The Biological Problem

In [ ]:
df = pd.read_csv("data/subcellular_localization/cleaned_dataset.csv")
df.head()

Our subcellular localization dataset shows us a protein sequence, the split of each data point, and 8 different locations in which this protein could be expressed. With this dataset, our goal is to create a model that predicts where a protein sequence of our choice is likely to be expressed across these 8 classes. 

At first, it might seem like a simple goal, but how are we going to convert protein sequences into a numerical format to make these predictions? We can't just use the strings of the sequences as they are.

Also, you have previously seen how we can do binary classification with a sigmoidal activation function, but how are we meant to extend our prediction task to multiple classes?

We'll look into how to do both of these in the following sections of this notebook.

---
**Q*1: How many proteins are in our dataset and how are they distributed across the 8 different cellular locations in percentages?**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# YOUR CODE HERE


## Part 2: Load Embeddings

A machine learning model can't understand a protein sequence like "MAGAASPCPLRLP..." directly. We need to convert these sequences into a numerical format.

There are simple methods like one-hot encoding, but these don't capture the complex biochemical and evolutionary information embedded in the sequence. Instead, we'll use embeddings from a Protein Language Model (PLM) since this is what's currently used in research.

Models like ESM2 are massive neural networks trained on millions of known protein sequences. By learning the "language" of proteins, they can convert any amino acid sequence into a rich numerical vector. This is a powerful form of transfer learning.

We showed an example of how to acquire these embeddings in Week 8 Intro. To save time, we will load embeddings here instead of computing them since it can be computationally demanding.

In [ ]:
# Load the embeddings
X = np.load("data/subcellular_localization/embeddings.npy")

# Print the shape to confirm
print("Shape of embeddings matrix:", X.shape)
print(f"This means we have {X.shape[0]} proteins, each represented by a vector of {X.shape[1]} numbers.")

## Part 3: Visualizing High Dimensional Data

In [ ]:
from sklearn.decomposition import PCA

# Reduce dimensions from 320 to 2 using PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(X)

# Prepare labels for coloring
label_cols = ["Cell membrane", "Cytoplasm", "Endoplasmic reticulum", "Golgi apparatus", "Lysosome/Vacuole", "Mitochondrion", "Nucleus", "Peroxisome"]
# Convert one-hot encoded labels to a single integer label per protein
y_integers = np.argmax(df[label_cols].values, axis=1)

# Create the plot
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    hue=[label_cols[i] for i in y_integers],
    palette='tab10',
    alpha=0.7,
    s=20
)

plt.title('PCA of Protein Embeddings Colored by Subcellular Location')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.legend(title='Location', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
**Q*2: Look at the PCA plot. Do you see any distinct clusters? Are some locations better separated than others? What might this imply about how easily a machine learning model can distinguish between these classes?**

<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Part 4: Data Preparation for Modeling

Before we can train any models, we need to prepare our data splits and format our labels correctly.

### Pre-processing Labels

The cleaned dataset contains the labels as separate columns. `PyTorch`'s `CrossEntropyLoss` expects a single list of integers (e.g., [0, 0, 0, 1, 0, 0]) corresponding to our classes. 

We need to convert our columns into a single target array. Since each column is assigned as a binary value, we can convert this to a list of intergers using `np.argmax`.

In [ ]:
# Create integer labels
label_cols = ["Cell membrane", "Cytoplasm", "Endoplasmic reticulum", "Golgi apparatus", "Lysosome/Vacuole", "Mitochondrion", "Nucleus", "Peroxisome"]
y = np.argmax(df[label_cols].values, axis=1) 
y.shape

Data splitting is all about managing data leakage. In biology, sequences are typically split by how similar they are to one another using tools like MMSeqs2. In our subcellular localization dataset, this has already been done for us. We just need to assign these splits to different variables to train our model.

We will first mask out each partition as follows.

In [ ]:
# Create boolean masks for each data split
train_mask = (df['Partition'] == 'train')
valid_mask = (df['Partition'] == 'valid')
test_mask = (df['Partition'] == 'test')

Next, we will use the masks we've just created to split our dataset into training, validation, and test sets.

In [ ]:
# Apply masks to get our final data splits
X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]
X_test, y_test = X[test_mask], y[test_mask]

Lastly, we will verify that our splits are appropriately sized.

In [ ]:
# Print to get an idea of the sizes
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_valid.shape, y_valid.shape)
print("Test set shape:", X_test.shape, y_test.shape)

## Part 5: `scikit-learn` Machine Learning Models

Before building complex neural networks, it's always good practice to establish a baseline with simpler machine learning models. Let's see how well models from `scikit-learn` perform on this task.

We'll start with a `LogisticRegression` model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize model
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Train model on training data
print("Training Logistic Regression model...")
log_reg.fit(X_train, y_train)

# Evaluate model on validation data
y_pred_log_reg = log_reg.predict(X_valid)
accuracy_log_reg = accuracy_score(y_valid, y_pred_log_reg)

print(f"Logistic Regression Validation Accuracy: {accuracy_log_reg:.4f}")

This gives us a solid baseline. Now, it's your turn to try a more powerful model.

## Part 6: Building a Neural Network in `PyTorch`

While classical models are fast, neural networks have the potential to learn more complex patterns from the embeddings. Let's build our pipeline in `PyTorch`.

### Create Dataset Class

First, we need a custom `Dataset` class to feed our data into `PyTorch`. This should be familiar from Week 3.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

We'll have the precomputed embeddings as input features and class labels as target outputs. We must ensure our inputs and outputs can be read by `PyTorch`, so we need to ensure we convert the `numpy` arrays from our saved embeddings to `PyTorch` tensors.

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        # Convert numpy arrays to PyTorch tensors
        self.embeddings = torch.tensor(embeddings)
        self.labels = torch.tensor(labels)

    def __len__(self):
        # Return length of dataset in number of samples
        return len(self.labels)

    def __getitem__(self, idx):
        # Return slice of a dataset to be used as a batch
        return self.embeddings[idx], self.labels[idx]

We'll need to create a `Dataset` for each split of our data as well.

In [ ]:
# Create Datasets
train_dataset = EmbeddingDataset(X_train, y_train)
valid_dataset = EmbeddingDataset(X_valid, y_valid)
test_dataset = EmbeddingDataset(X_test, y_test)

Lastly, we'll have to wrap each `Dataset` class with a `DataLoader` class to be able to work effectively in `PyTorch`'s training loop.

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Create Classifier Architecture

We will use the `FlexibleFullyConnectedNetwork` from Week 5 as our model's architecture.

In [ ]:
# Define dimensions
INPUT_DIM = X_train.shape[1]
NUM_CLASSES = len(label_cols)

class FlexibleFullyConnectedNetwork(nn.Module):
    def __init__(self, layer_dims, activation=nn.ReLU):
        super(FlexibleFullyConnectedNetwork, self).__init__()

        # Creating the layers using ModuleList
        layers = nn.ModuleList()

        # Hidden layers
        for i in range(len(layer_dims) - 2):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))  # Fully connected layer
            layers.append(activation())                                 # Activation function

        # Output layer
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))        # Output layer

        # Store the layers in the class
        self.layers = layers

    def forward(self, x):
        # Pass x through all layers
        for layer in self.layers:
            x = layer(x)

        return x

For our purposes, we'll need to collect the input and output dimensions from our data.

In [ ]:
# Define dimensions
INPUT_DIM = X_train.shape[1]
NUM_CLASSES = len(label_cols)

# Instantiate a simple model
model = FlexibleFullyConnectedNetwork([INPUT_DIM, 128, NUM_CLASSES])
print(model)

You should expect to see:

```
FlexibleFullyConnectedNetwork(
  (layers): ModuleList(
    (0): Linear(in_features=320, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=8, bias=True)
  )
)
```

### Training Loop

Here is a standard training and evaluation loop for `PyTorch`. This function will train the model for a specified number of epochs and evaluate its performance on the validation set after each epoch.

In [ ]:
def train_and_evaluate(model, train_loader, valid_loader, epochs=10, lr=0.001):
    # Use either cpu or gpu depending on what's avaialble
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Set model to device
    model.to(device)
    
    # Initialize loss and optimimzer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Initialize loss tracker
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    
    print(f"Starting training on {device}...")
    # Iterate through epochs
    for epoch in range(epochs):
        # Activate update calculations (update optimzer on train data)
        model.train()
        train_loss = 0.0

        # Iterate through train batches
        for inputs, labels in train_loader:
            # Set input/output tensors to device
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Clear optimizer updates
            optimizer.zero_grad()

            # Get predictions from current model
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs, labels)

            # Calculate updates from backward pass
            loss.backward()

            # Update model weights from backward pass
            optimizer.step()

            # Add train loss for batch
            train_loss += loss.item() * inputs.size(0)
        
        # Deactivate update calculations (no update optimzer on valid/test data)
        model.eval()
        val_loss, correct, total = 0.0, 0, 0

        # Prevent torch from calculating updates from backward pass
        with torch.no_grad():
            # Iterate through valid batches
            for inputs, labels in valid_loader:
                # Set input/output tensors to device
                inputs, labels = inputs.to(device), labels.to(device)

                # Get predictions from current model
                outputs = model(inputs)

                # Calculate loss
                loss = criterion(outputs, labels)

                # Update valid loss for batch
                val_loss += loss.item() * inputs.size(0)
                
                # Update valid accuracy for batch
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        # Update metrics for entire epoch
        avg_train_loss = train_loss / len(train_loader.dataset)
        avg_val_loss = val_loss / len(valid_loader.dataset)
        val_accuracy = correct / total
        
        # Add metrics to loss tracker
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_accuracy'].append(val_accuracy)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
    print("Training finished.")
    return history

We will reuse the training logic from Week 4, except in this case, we don't use `MSELoss` (mean squared error loss) because we aren't performing a regression prediction task. Instead, we use `CrossEntropyLoss` for multiclass classification prediction tasks. The `Adam` optimizer can be used across any task since it's an independent building block to find the parameters (not the hyperparameters!) for our model that minimize the loss.

We'll also use the `evaluate_pytorch` function, which will come in handy when we want to see how our model performs on the test set in our final evaluation step.

In [ ]:
def evaluate_pytorch(model, data_loader):
    # Use either cpu or gpu depending on what's avaialble
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Set model to device
    model.to(device)

    # Deactivate update calculations (no update optimzer on valid/test data)
    model.eval()

    # Initialize loss
    criterion = nn.CrossEntropyLoss()
    
    # Initialize trackers
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_predictions = [], []
    
    # Prevent torch from calculating updates from backward pass
    with torch.no_grad():
        # Iterate through valid batches
        for inputs, labels in valid_loader:
            # Set input/output tensors to device
            inputs, labels = inputs.to(device), labels.to(device)

            # Get predictions
            outputs = model(inputs)
            
            # Calculate loss
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Store labels and predictions for confusion matrix
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
            
    avg_loss = running_loss / len(data_loader.dataset)
    accuracy = correct / total
    
    return avg_loss, accuracy, all_labels, all_predictions

## **Graded Exercises: (20 marks)**

This is your capstone project. You will train several models, log your results, and analyze their performance to find the best one.

**GQ*1 (4 marks): Train a Random Forest Classifier.**

**Train a `RandomForestClassifier` from `sklearn` on the training data. Report its accuracy on the validation set.**

<span style="background-color: #FFD700">**Write your code below**</span>

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# YOUR CODE HERE
# Initialize model

# Train model on training data

# Evaluate on validation data

# Report final validation accuracy


**GQ*2 (4 marks): Train a Baseline Neural Network.**

**Instantiate and train a simple neural network with one hidden layer of 128 neurons. Train it for 15 epochs with a learning rate of 0.001. Log the final validation accuracy.**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# YOUR CODE HERE
# Instantiate model

# Train and evaluate model

# Report final validation accuracy


**GQ*3 (6 marks): Design and Train Your Own Neural Network.**

**Now, it's time for you to experiment! Your goal is to design a model that outperforms the baseline.**

**You can change:**
- **Architecture: Make it deeper (e.g., `[INPUT_DIM, 256, 128, NUM_CLASSES]`) or wider.**
- **Hyperparameters: Adjust the learning rate or the number of epochs.**

**Train at least one new model with your custom settings.**


<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# YOUR CODE HERE
# A deeper model with a slightly lower learning rate

# Instantiate model

# Train and evaluate model


**GQ*4 (6 marks): Final Evaluation and Interpretation.**

**Choose your best performing model from all the experiments you've run (Logistic Regression, Random Forest, Baseline NN, Custom NN).**
- **Evaluate models on test set: Report its final accuracy on the test set (unseen data).**
- **Plot confusion matrix: Generate and plot a confusion matrix for its predictions on the test set.**
- **Interpret results: In 2-3 sentences, comment on your results. Which classes were the most difficult for your model to predict? Why might that be the case, referencing the PCA plot or biological intuition?**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix(y_true, y_pred):
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_cols)

    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(10, 10))
    disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation='vertical')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# YOUR CODE HERE
# Use best model based on validation performance

# Evaluate on test set using evaluate_pytorch() for pytorch model or model.predict

# Plot confusion matrix


<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Conclusion

Congratulations on completing the capstone project! You have successfully navigated a full machine learning pipeline in bioinformatics. You started with raw biological data, leveraged powerful pretrained embeddings, explored the data visually, established performance baselines with classical models, and engineered your own neural network.

The process of experimenting, logging results, and interpreting performance is the core daily work of a machine learning scientist. The skills you've consolidated here are fundamental and will serve you well in any future data science endeavour.